> 📄 English: [langsmith_basics.ipynb](langsmith_basics.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--tracing-with-langsmith--langsmith-basics)

# LangSmith 教程：用 LangGraph 为 AI 系统添加可观测性

## 概览

本教程演示如何使用 LangSmith 和 LangGraph 为 AI 系统添加可观测性。虽然构建 AI 应用已经变得越来越容易，但理解它们如何做决策，以及监控它们在真实使用中的行为，仍然颇具挑战。

### 为什么可观测性很重要

大多数 AI 应用在演示中表现良好，但一旦部署，调试和优化就变得困难。由于看不到决策过程，团队会在一些基本问题上苦苦挣扎：AI 为什么选择了这个特定的响应？系统的哪些部分又慢又贵？我们如何基于真实使用模式系统地改进性能？哪些工具调用路径最有效？这些路径中哪些是最有效、最可复用且成本最低的？

可以把可观测性想象成给你的 AI 系统加装一个“飞行记录仪”。正如航空公司用黑匣子来了解飞行过程中发生了什么，LangSmith 会捕获你 AI 系统中的每一个决策、时间点以及数据流。这把 AI 开发从“猜”变成了“工程”。

### 你将学到什么

跟随本教程，你将理解如何对 AI 系统进行埋点，以实现透明性和可观测性。

我们将使用 LangGraph 构建一个简单的研究助手，展示关键的可观测性模式。该助手会分析问题、决定是否使用工具、在需要时执行这些工具，并给出有用的回复。在整个过程中，LangSmith 会捕获详细的追踪（trace），展示每一个决策点、时间数据以及信息在系统中的流动。


## 前置条件与初始配置

在开始构建之前，我们需要配置开发环境。这包括安装必要的包，以及为 OpenAI（驱动我们的 AI）和 LangSmith（提供可观测性）配置 API key。

理解这个配置至关重要，因为 LangSmith 的工作原理是自动拦截并记录所有 LangGraph 操作。一旦配置完成，每一次 LLM 调用、工具执行和工作流步骤都会被捕获，而无需额外的代码改动。

要求：
- Python 3.9+
- OpenAI API key（[在这里获取](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--tracing-with-langsmith--langsmith-basics&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fapi-keys&text=get%20one%20here)）
- LangSmith 账户（[免费注册](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--tracing-with-langsmith--langsmith-basics&click=langchain&target=https%3A%2F%2Fsmith.langchain.com&text=free%20signup)）- 它提供了可观测性仪表盘，你将在那里看到所有洞察


In [ ]:
# Install required packages for AI development with observability
!pip install -U langchain-core langchain-openai langgraph langsmith requests

## API 配置

现在我们来配置 API key 并启用 LangSmith 追踪。这里的关键在于，设置 `LANGCHAIN_TRACING_V2=true` 会自动启用对所有操作的全面日志记录。可以把它想象成为你的 AI 系统安装一个“飞行记录仪”。从现在起，每一个决策和操作都会被捕获，并显示在你的 LangSmith 仪表盘中。

这与传统日志记录有着根本区别，因为 LangSmith 理解 AI 工作流的结构。它不只是捕获文本日志，而是构建出信息在你系统中如何流动的完整图景。


In [ ]:
import os

# Configure API keys - replace with your actual keys
os.environ['OPENAI_API_KEY'] = ''
os.environ['LANGCHAIN_API_KEY'] = ''
os.environ['LANGCHAIN_TRACING_V2'] = 'true'  # This triggers observability
os.environ['LANGCHAIN_PROJECT'] = 'langsmith-tutorial-demo' #This is the project name where the traces will be stored

# Verify configuration
required_vars = ['OPENAI_API_KEY', 'LANGCHAIN_API_KEY']
for var in required_vars:
    if not os.getenv(var) or 'your_' in os.getenv(var, ''):
        print(f"Warning: {var} needs your actual key")
    else:
        print(f"✓ {var} configured")

print(f"\nLangSmith Project: {os.getenv('LANGCHAIN_PROJECT')}")
print("\nTracing is now active - all AI operations will be logged for analysis")
print("Visit https://smith.langchain.com to see your traces")

## 构建一个简单的可观测 Agent

我们将创建一个最小系统，但仍然足以展示 LangSmith 可观测性的威力。我们的 Agent 只有两个能力：直接从训练数据回答问题，以及在需要最新信息时使用一个简单的搜索工具。

让我们从配置基本组件开始：


In [ ]:
from typing import TypedDict, List
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
import requests
import time
import json

# Initialize the language model with deterministic settings
# Using temperature=0 ensures consistent responses, making it easier to analyze patterns
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Language model initialized with temperature=0 for consistent behavior")
print("All LLM calls will be automatically traced in LangSmith")

## 定义我们简单的 Agent 状态

Agent 的状态代表流经我们工作流的信息。对于这个简单 Agent，我们只需要刚好足够的状态来展示有意义的可观测性。可以把它想象成 LangSmith 在流程的每一步都会跟踪的“记忆”。

状态中的每个字段在可观测性方面都有特定用途。user_question 字段让我们把行为与输入类型关联起来。needs_search 字段展示 Agent 的决策过程。search_result 字段捕获工具执行结果。reasoning 字段则提供了明确的解释，帮助我们理解 Agent 为什么做出特定选择。


In [ ]:
class AgentState(TypedDict):
    """Simple state that flows through our agent workflow."""
    user_question: str        # The original question from the user
    needs_search: bool        # Whether we determined search is needed
    search_result: str        # Result from our search tool (if used)
    final_answer: str         # The response we'll give to the user
    reasoning: str            # Why we made our decisions (great for observability)

print("Agent state schema defined")
print("This structured state enables LangSmith to track information flow")

## 创建我们的搜索工具

我们将实现一个可以从 Wikipedia 获取最新信息的搜索函数。这个工具演示了 LangSmith 如何捕获工具执行细节，包括时间、成功/失败状态以及返回的数据。

该实现正确使用了 Wikipedia 的搜索 API。与要求精确页面标题的页面摘要端点不同，搜索 API 可以处理通用查询并返回相关结果。请注意我们是如何包含全面的错误处理和日志语句的。这些都会出现在你的 LangSmith 追踪中，帮助你理解执行过程中发生了什么。

理解摘要 API 与搜索 API 的区别很重要：摘要 API 期望精确的页面标题（如 "Artificial_intelligence"），而搜索 API 可以处理自然语言查询（如 "what is AI"）。


In [ ]:
@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for current information about a topic."""
    try:
        # Use Wikipedia's proper search API that can handle general queries
        # This is different from the page summary API which requires exact page titles
        search_url = "https://en.wikipedia.org/w/api.php"
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": 3  # Get top 3 results
        }
        
        response = requests.get(search_url, params=search_params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            search_results = data.get('query', {}).get('search', [])
            
            if search_results:
                # Get the most relevant result and fetch its summary
                top_result = search_results[0]
                page_title = top_result['title']
                
                # Now get the page summary using the exact title
                summary_url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{page_title.replace(' ', '_')}"
                summary_response = requests.get(summary_url, timeout=10)
                
                if summary_response.status_code == 200:
                    summary_data = summary_response.json()
                    extract = summary_data.get('extract', 'No summary available')
                    # Truncate for readability in traces
                    return f"Found information about '{page_title}': {extract[:400]}..."
                else:
                    return f"Found '{page_title}' but couldn't retrieve summary"
            else:
                return f"No Wikipedia articles found for '{query}'"
        else:
            return f"Wikipedia search failed with status {response.status_code}"
            
    except Exception as e:
        # This error handling will show up in LangSmith traces
        return f"Search error: {str(e)}"

print("Search tool created with proper Wikipedia search API integration")
print("Tool execution timing and results will be captured automatically")

## 构建决策工作流

现在我们来创建 Agent 的核心逻辑。虽然这是一个简单系统，我们会把它组织成独立的函数，以演示 LangSmith 如何追踪多步骤工作流。每个函数都代表一个清晰的决策点，会显示在你的可观测性仪表盘中。

这种模块化方法有两个目的：让代码更易理解和测试，并创建 LangSmith 可以捕获和分析的自然断点。

### 第 1 步：决定是否搜索

第一步分析用户的问题，并决定我们是需要搜索最新信息，还是可以直接从模型的训练数据中回答。


In [ ]:
def decide_search_need(state: AgentState) -> AgentState:
    """Analyze the question and decide if we need to search for current information."""
    user_question = state["user_question"]
    
    # This prompt engineering is visible in LangSmith traces
    # Notice how we're asking for a structured response to make parsing reliable
    decision_prompt = f"""
    Analyze this question and decide if it requires current/recent information that might not be in your training data:
    
    Question: "{user_question}"
    
    Consider:
    - Does this ask about recent events, current prices, or breaking news?
    - Does this ask about people, companies, or topics that change frequently?
    - Can you answer this well using your existing knowledge?
    
    Respond with exactly "SEARCH" if you need current information, or "DIRECT" if you can answer directly.
    Then on a new line, briefly explain your reasoning.
    """
    
    response = llm.invoke([SystemMessage(content=decision_prompt)])
    decision_text = response.content.strip()
    
    # Parse the response - this parsing logic will be visible in traces
    lines = decision_text.split('\n')
    decision = lines[0].strip()
    reasoning = lines[1] if len(lines) > 1 else "No reasoning provided"
    
    # Update state with our decision
    state["needs_search"] = decision == "SEARCH"
    state["reasoning"] = f"Decision: {decision}. Reasoning: {reasoning}"
    
    # This print statement will help you see the flow during execution
    print(f"Decision: {'SEARCH' if state['needs_search'] else 'DIRECT'} - {reasoning}")
    
    return state

### 第 2 步：在需要时执行搜索

如果上一步确定需要搜索，这个函数就会执行我们的搜索工具。LangSmith 不仅显示搜索是否发生，还显示发送了什么样的查询、花费了多长时间以及返回了什么结果。

这种条件执行模式在 AI 系统中很常见，LangSmith 通过向你展示走了哪条路径以及为什么来处理它。


In [ ]:
def execute_search(state: AgentState) -> AgentState:
    """Execute search if needed, otherwise skip this step."""
    if not state["needs_search"]:
        print("Skipping search - not needed for this question")
        state["search_result"] = "No search performed"
        return state
    
    print(f"Executing search for: {state['user_question']}")
    
    # Execute our search tool - this will show up as a separate step in LangSmith
    # The .invoke() call will be traced with full input/output details
    search_result = wikipedia_search.invoke({"query": state["user_question"]})
    state["search_result"] = search_result
    
    print(f"Search completed: {len(search_result)} characters returned")
    
    return state

### 第 3 步：生成最终响应

最后一步把所有可用信息综合成一个有用的响应。在这里，我们把模型内置的知识与我们收集到的任何搜索结果结合起来。综合过程往往是 AI 系统中最复杂的部分，而 LangSmith 的可观测性帮助你理解这个过程的效果如何。


In [ ]:
def generate_response(state: AgentState) -> AgentState:
    """Generate the final response using all available information."""
    user_question = state["user_question"]
    search_result = state.get("search_result", "")
    used_search = state["needs_search"]
    
    # Build context for the response
    # This conditional logic creates different prompt patterns that LangSmith will capture
    if used_search and "Search error" not in search_result:
        context = f"Question: {user_question}\n\nSearch Results: {search_result}"
        response_prompt = f"""
        Answer the user's question using both your knowledge and the search results provided.
        
        {context}
        
        Provide a helpful, accurate response that synthesizes the information.
        """
    else:
        response_prompt = f"""
        Answer this question using your existing knowledge:
        
        {user_question}
        
        Provide a helpful, accurate response.
        """
    
    # This LLM call will be traced with the complete prompt and response
    response = llm.invoke([SystemMessage(content=response_prompt)])
    state["final_answer"] = response.content
    
    print(f"Response generated: {len(response.content)} characters")
    
    return state

## 组装工作流

现在我们将用 LangGraph 把各个函数连接成一个完整的工作流。这会创建一个 LangSmith 可以可视化的显式图结构，向你展示 AI 在决策过程中经过的确切路径。

图结构对可观测性尤其有价值，因为它让控制流变得显式。LangGraph 不是把条件逻辑埋在函数调用中，而是创建一种可视化表示，精确展示哪些步骤被执行以及以什么顺序执行。

可以把这个过程想象成创建一张路线图，LangSmith 可以照着它向你展示数据在系统中经历过的旅程。


In [ ]:
# Build the workflow graph
workflow = StateGraph(AgentState)

# Add our three workflow steps
# Each of these will appear as a distinct node in LangSmith's graph visualization
workflow.add_node("decide", decide_search_need)
workflow.add_node("search", execute_search)
workflow.add_node("respond", generate_response)

# Define the flow with conditional logic
# LangSmith will show you which edges were traversed for each execution
workflow.set_entry_point("decide")
workflow.add_edge("decide", "search")     # Always go to search step (it will skip if not needed)
workflow.add_edge("search", "respond")    # Then generate response
workflow.add_edge("respond", END)         # Finish

# Compile into an executable agent
simple_agent = workflow.compile()

print("Workflow compiled successfully")
print("Flow: decide → search → generate_response")
print("Ready to demonstrate LangSmith observability")

## 工作流可视化

下面是我们的 Agent 工作流图的可视化。此图展示了 LangSmith 在执行期间将追踪的结构：

<img src="assets/wiki_agent_td.png" alt="Agent Workflow Graph" width="700">

这个可视化帮助我们理解 Agent 各组件之间的流程：决策、搜索执行和响应生成。


## 在完整可观测性下进行测试

我们将在不同类型的问题上运行我们的 Agent，看看 LangSmith 如何捕获执行的每一个细节。每次测试都会生成一条完整的追踪，向你展示决策过程、时间信息以及所有中间结果。

这个测试运行器包含时间测量和元数据标记，有助于在 LangSmith 中组织你的追踪，方便分析。元数据和标签充当过滤器，让你对相似类型的执行进行分组和比较。


In [ ]:
def run_test_with_observability(question: str, test_type: str) -> dict:
    """Run a test and capture comprehensive observability data."""
    print(f"\n{'='*60}")
    print(f"Testing: {question}")
    print(f"Type: {test_type}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    # Initialize state for this test
    initial_state = {
        "user_question": question,
        "needs_search": False,
        "search_result": "",
        "final_answer": "",
        "reasoning": ""
    }
    
    try:
        # Execute with metadata for LangSmith
        # This metadata will help you filter and analyze traces later
        config = {
            "metadata": {
                "test_type": test_type,
                "tutorial": "langsmith-observability"
            },
            "tags": ["tutorial", "demo", test_type]
        }
        
        # This invoke call will create a complete trace in LangSmith
        final_state = simple_agent.invoke(initial_state, config=config)
        
        end_time = time.time()
        total_time = end_time - start_time
        
        # Display results for immediate feedback
        print(f"\nResults:")
        print(f"   Decision Process: {final_state['reasoning']}")
        print(f"   Used Search: {'Yes' if final_state['needs_search'] else 'No'}")
        print(f"   Response Length: {len(final_state['final_answer'])} characters")
        print(f"   Total Time: {total_time:.2f} seconds")
        print(f"\nAnswer: {final_state['final_answer'][:200]}...")
        
        return {
            "question": question,
            "type": test_type,
            "success": True,
            "used_search": final_state['needs_search'],
            "total_time": round(total_time, 2),
            "reasoning": final_state['reasoning']
        }
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return {
            "question": question,
            "type": test_type,
            "success": False,
            "error": str(e)
        }

## 运行我们的测试套件

让我们用应该触发不同行为的问题来测试我们的 Agent。我们有三类问题，它们将帮助我们理解 Agent 如何做决策：

**直接回答（Direct Answer）**：模型可以从训练数据中回答的问题。这些应该展示 Agent 选择不搜索，体现高效的资源使用。

**当前信息（Current Info）**：需要最新信息的问题。这些应该触发搜索行为，展示 Agent 如何处理随时间变化的信息。

**事实查询（Factual Lookup）**：关于既定事实、可能受益于核实的问题。这些将展示 Agent 如何在训练数据的置信度与当前核实的价值之间取得平衡。

观察每类问题如何以不同的方式流经系统，并关注 LangSmith 捕获的决策过程。


In [ ]:
# Define our test cases
test_cases = [
    {
        "question": "What is the capital of France?",
        "type": "direct_answer",
        "expected_search": False
    },
    {
        "question": "What happened in the 2024 US presidential election?",
        "type": "current_info",
        "expected_search": True
    },
    {
        "question": "Tell me about artificial intelligence",
        "type": "factual_lookup",
        "expected_search": False  # Should be answerable directly
    }
]

print("Starting LangSmith Observability Demo")
print("Each test will generate detailed traces in your LangSmith dashboard")
print("Visit https://smith.langchain.com to see real-time traces\n")

test_results = []

for i, test_case in enumerate(test_cases, 1):
    print(f"\nRunning Test {i}/{len(test_cases)}")
    
    result = run_test_with_observability(
        test_case["question"], 
        test_case["type"]
    )
    
    test_results.append(result)
    
    # Small delay to make the traces easier to distinguish in LangSmith
    time.sleep(1)

print(f"\n\nAll tests completed")
print(f"Generated {len(test_results)} traces in LangSmith")
print(f"Check your dashboard to explore the detailed execution data")

## 理解你的 LangSmith 仪表盘

现在你已经运行了测试，让我们来看看 LangSmith 捕获到了什么。导航到 [smith.langchain.com](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--tracing-with-langsmith--langsmith-basics&click=langchain&target=https%3A%2F%2Fsmith.langchain.com&text=smith.langchain.com) 上的 LangSmith 仪表盘，并选择 `langsmith-tutorial-demo` 项目。

### 你将在 LangSmith 中看到什么

**追踪列表视图（Trace List View）**：你会看到所有测试执行的列表。每一行代表你的 Agent 的一次完整运行，显示输入问题、执行时间、成功/失败状态以及总成本。这个视图让你对不同类型查询的系统性能有一个高层次的概览。

![本地图片](./assets/1.png)

可以把它想象成你系统的活动日志，但比传统日志的信息丰富得多。你可以按执行时间排序来查找慢查询，按标签过滤来分析特定类型的问题，或寻找每次查询成本中的规律。

**单条追踪详情（Individual Trace Details）**：点击任意一条追踪即可查看完整的执行流程。你会看到：

![本地图片](./assets/2.png)
**图可视化**：你工作流的可视化表示，展示执行了哪些节点以及数据如何在它们之间流动。这尤其强大，因为你可以一目了然地看到是否使用了搜索以及每一步花费了多长时间。

**逐步执行**：每次函数调用都有输入、输出和时间信息。这种细粒度视图帮助你不仅理解发生了什么，还理解为什么发生。你可以看到发送给语言模型的确切提示词，以及它给出的推理。

![本地图片](./assets/3.png)

**LLM 调用**：发送给语言模型的每一个提示词以及确切的响应。这种透明性对于提示词优化和理解模型行为至关重要。

**工具执行**：你的搜索工具是何时以及如何被调用的，包括发送的查询、收到的响应和执行时间。

**性能分析**：LangSmith 会自动聚合你所有运行的性能数据：

**延迟模式**：哪些步骤一直耗时最长？搜索总是瓶颈吗，还是决策有时会拖慢速度？

**成本分析**：每种类型的查询成本是多少？搜索查询是否明显比直接回答更贵？

**成功率**：是否有某些类别的查询持续失败？某些问题模式是否会导致搜索错误？

**工具使用模式**：搜索实际被需要的频率有多高？是否有些问题不必要地触发了搜索？


## 关键的可观测性洞察

基于你刚生成的追踪，以下是 LangSmith 所支持的几种洞察类型。这些洞察会改变你理解和改进 AI 系统的方式。

### 决策透明性

对于每个问题，你都能确切看到 Agent 为什么选择搜索或直接回答。这种透明性对于调试意外行为以及建立对 AI 系统的信任至关重要。当用户问系统为什么以某种方式表现时，你可以指向追踪中捕获的具体推理。

### 性能优化机会

通过比较不同问题类型的执行时间，你可以识别瓶颈和优化机会。例如，你可能会发现搜索查询明显比直接回答更耗时，这暗示着缓存或并行执行的机会。

### 成本管理

LangSmith 会显示每次 LLM 调用的 token 使用量和估算成本。这种细粒度的成本数据帮助你以没有可观测性就无法实现的方式进行优化。你可以识别昂贵的操作，并优化提示词或路由逻辑，在不牺牲质量的前提下降低成本。

### 质量保证

有了完整的追踪，你可以验证 Agent 是否始终做出合理的决策。你可以发现成功与失败执行之间的模式，识别需要额外处理的边界情况，并根据真实使用模式创建回归测试。

这种系统化的质量保证方法，只有在全面的可观测性下才可能实现。


## 更进一步

本教程用一个简单 Agent 演示了 AI 可观测性的基本原则。你学到的模式可以扩展到复杂得多的系统。以下是一些入门建议：

### 用于你自己的项目

**从第一天开始就引入可观测性**：在构建复杂功能之前就启用 LangSmith 追踪。

**结构化你的工作流**：使用 LangGraph 的显式工作流结构，让决策点可见。

**添加有意义的元数据**：为你的追踪打上业务上下文标签，以便更好地分析。

**监控关键指标**：基于你在 LangSmith 中观察到的模式，为延迟、成本和错误率设置告警。

你在本教程中建立的可观测性基础，在生产环境中会变得更有价值，因为理解系统行为对于维护服务质量和用户信任至关重要。


---
本教程作者：[Shivnarayan Rajappa](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--tracing-with-langsmith--langsmith-basics&click=linkedin-shivnarayanrajappa&target=https%3A%2F%2Fwww.linkedin.com%2Fin%2Fshivnarayanrajappa%2F&text=Shivnarayan%20Rajappa)
